In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression

print("所有库导入成功！")
print(f"Pandas 版本: {pd.__version__}")
print(f"NumPy 版本: {np.__version__}")

所有库导入成功！
Pandas 版本: 2.3.3
NumPy 版本: 2.3.5


In [2]:
# 1. 读取数据
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

# 2. 查看数据基本信息
print("训练集形状:", train.shape)
print("测试集形状:", test.shape)
print("\n训练集前5行:")
print(train.head())

# 3. 查看列名和数据类型
print("\n训练集信息:")
print(train.info())

训练集形状: (891, 12)
测试集形状: (418, 11)

训练集前5行:
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0    

In [3]:
# 1. 选择特征列
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare']

# 2. 处理缺失值（用中位数填充 Age，用均值填充 Fare）
train['Age'] = train['Age'].fillna(train['Age'].median())
test['Age'] = test['Age'].fillna(test['Age'].median())
test['Fare'] = test['Fare'].fillna(test['Fare'].median())

# 3. 将性别转换为数字（男=0，女=1）
train['Sex'] = train['Sex'].map({'male': 0, 'female': 1})
test['Sex'] = test['Sex'].map({'male': 0, 'female': 1})

# 4. 准备训练数据
X = train[features]
y = train['Survived']

# 5. 训练逻辑回归模型
model = LogisticRegression(max_iter=1000)
model.fit(X, y)

print("模型训练完成！")
print(f"特征重要性（系数）: {dict(zip(features, model.coef_[0]))}")

模型训练完成！
特征重要性（系数）: {'Pclass': np.float64(-1.051465775059578), 'Sex': np.float64(2.6503450743166916), 'Age': np.float64(-0.03851175501496326), 'SibSp': np.float64(-0.33687472275836633), 'Parch': np.float64(-0.09600862727792073), 'Fare': np.float64(0.003011995769369435)}


In [4]:
# 6. 用模型预测测试集
predictions = model.predict(test[features])

# 7. 生成提交文件
output = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': predictions
})

# 保存为 CSV
output.to_csv('submission.csv', index=False)

print("提交文件已生成！")
print(f"预测结果形状: {output.shape}")
print("\n前10行预览:")
print(output.head(10))

# 统计预测结果
print(f"\n预测存活人数: {output['Survived'].sum()}")
print(f"预测死亡人数: {len(output) - output['Survived'].sum()}")

提交文件已生成！
预测结果形状: (418, 2)

前10行预览:
   PassengerId  Survived
0          892         0
1          893         0
2          894         0
3          895         0
4          896         1
5          897         0
6          898         1
7          899         0
8          900         1
9          901         0

预测存活人数: 161
预测死亡人数: 257


In [5]:
# 1. 从 Name 中提取头衔（Mr, Mrs, Miss, Master 等）
train['Title'] = train['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
test['Title'] = test['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)

# 查看头衔分布
print("训练集头衔分布：")
print(train['Title'].value_counts())

# 2. 合并稀有头衔
title_mapping = {
    'Mr': 'Mr', 'Mrs': 'Mrs', 'Miss': 'Miss', 'Master': 'Master',
    'Dr': 'Rare', 'Rev': 'Rare', 'Col': 'Rare', 'Major': 'Rare',
    'Mlle': 'Miss', 'Countess': 'Rare', 'Ms': 'Miss', 'Lady': 'Rare',
    'Jonkheer': 'Rare', 'Don': 'Rare', 'Dona': 'Rare', 'Mme': 'Mrs',
    'Capt': 'Rare', 'Sir': 'Rare'
}

train['Title'] = train['Title'].map(title_mapping)
test['Title'] = test['Title'].map(title_mapping)

# 头衔转数字
title_map = {'Mr': 0, 'Miss': 1, 'Mrs': 2, 'Master': 3, 'Rare': 4}
train['Title'] = train['Title'].map(title_map)
test['Title'] = test['Title'].map(title_map)

print("\n头衔映射后：")
print(train['Title'].value_counts())

训练集头衔分布：
Title
Mr          517
Miss        182
Mrs         125
Master       40
Dr            7
Rev           6
Mlle          2
Major         2
Col           2
Countess      1
Capt          1
Ms            1
Sir           1
Lady          1
Mme           1
Don           1
Jonkheer      1
Name: count, dtype: int64

头衔映射后：
Title
0    517
1    185
2    126
3     40
4     23
Name: count, dtype: int64


In [6]:
# 3. 创建家庭人数特征
train['FamilySize'] = train['SibSp'] + train['Parch'] + 1
test['FamilySize'] = test['SibSp'] + test['Parch'] + 1

# 4. 是否独自一人
train['IsAlone'] = (train['FamilySize'] == 1).astype(int)
test['IsAlone'] = (test['FamilySize'] == 1).astype(int)

# 5. 按性别和等级分组填充 Age（更精确）
# 先填充缺失值，用各组的平均值
train['Age'] = train.groupby(['Sex', 'Pclass'])['Age'].transform(
    lambda x: x.fillna(x.median())
)
test['Age'] = test.groupby(['Sex', 'Pclass'])['Age'].transform(
    lambda x: x.fillna(x.median())
)

# 6. 票价按等级分组填充
test['Fare'] = test.groupby('Pclass')['Fare'].transform(
    lambda x: x.fillna(x.median())
)

print("特征工程完成！")
print(f"新特征列: Title, FamilySize, IsAlone")
print(f"\n训练集新特征预览:")
print(train[['Title', 'FamilySize', 'IsAlone', 'Age', 'Fare']].head())

特征工程完成！
新特征列: Title, FamilySize, IsAlone

训练集新特征预览:
   Title  FamilySize  IsAlone   Age     Fare
0      0           2        0  22.0   7.2500
1      2           2        0  38.0  71.2833
2      1           1        1  26.0   7.9250
3      2           2        0  35.0  53.1000
4      0           1        1  35.0   8.0500


In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# 更新特征列表
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Title', 'FamilySize', 'IsAlone']

# 准备数据
X = train[features]
y = train['Survived']

# 随机森林模型
rf_model = RandomForestClassifier(
    n_estimators=100,      # 100棵树
    max_depth=5,           # 树的最大深度
    random_state=42,       # 随机种子，保证可复现
    min_samples_split=10,  # 内部节点再划分所需最小样本数
    min_samples_leaf=5     # 叶子节点最小样本数
)

# 交叉验证评估
cv_scores = cross_val_score(rf_model, X, y, cv=5)
print(f"交叉验证准确率: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

# 训练模型
rf_model.fit(X, y)

# 预测
predictions = rf_model.predict(test[features])

# 生成提交文件
output = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': predictions
})

output.to_csv('submission_v2.csv', index=False)

print("\n新提交文件已生成: submission_v2.csv")
print(f"预测存活人数: {output['Survived'].sum()}")
print(f"预测死亡人数: {len(output) - output['Survived'].sum()}")

# 特征重要性
importances = pd.DataFrame({
    'feature': features,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\n特征重要性:")
print(importances)

交叉验证准确率: 0.8260 (+/- 0.0134)

新提交文件已生成: submission_v2.csv
预测存活人数: 164
预测死亡人数: 254

特征重要性:
      feature  importance
1         Sex    0.313484
6       Title    0.276865
0      Pclass    0.126684
5        Fare    0.107821
2         Age    0.065478
7  FamilySize    0.061747
3       SibSp    0.028515
4       Parch    0.010783
8     IsAlone    0.008624
